# 04 – DQ Export (CSV for SharePoint)
Builds a business-readable file of the **current** failed records (latest run) and writes it to the Lakehouse:

- `Files/dq_exports/latest/DQ_Failed_Records.csv` – same name every day; the pipeline copies this to SharePoint
- `Files/dq_exports/archive/DQ_Failed_Records_<date>.csv` – dated copy, kept `EXPORT_ARCHIVE_DAYS` days

Run it in the pipeline **after** `03_dq_runner`. It does not change any DQ table.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: customer details for the file  (EDIT the columns to what the business needs)
# Joined only to rules whose entity_grain = 'CUSTOMER' (record_key = cust_account_id).
# Rules at other grains still appear in the file, with these columns empty.
customer_details = spark.sql(f"""
    SELECT CAST(ca.cust_account_id AS STRING) AS record_key,
           ca.account_number                  AS customer_account_number,
           p.party_number                     AS party_number,
           p.party_name                       AS customer_name,
           ca.status                          AS account_status
    FROM {SRC}.hz_cust_accounts ca
    JOIN {SRC}.hz_parties p ON p.party_id = ca.party_id
""")

In [ ]:
# Step 2: build the export dataset
rules = (spark.table(T_CATALOG)
              .select("rule_id", "rule_name", "rule_description", "dimension_name",
                      "severity", "entity_grain"))

failed = spark.table(T_FAILED_CURR)

export_df = (failed.join(rules, "rule_id", "left")
                   .join(customer_details.withColumn("entity_grain", F.lit("CUSTOMER")),
                         ["record_key", "entity_grain"], "left")
                   .select(F.col("run_date").alias("dq_run_date"),
                           "rule_id", "rule_name", "rule_description",
                           F.col("dimension_name").alias("dq_dimension"),
                           "severity", "entity_grain", "record_key",
                           "customer_account_number", "party_number", "customer_name", "account_status",
                           "first_failed_date", "days_failing")
                   .orderBy("rule_id", "record_key"))

pdf = export_df.toPandas()   # fine up to a few hundred thousand rows
print(f"Rows to export: {len(pdf)}")

In [ ]:
# Step 3: write the latest file + a dated archive copy
import os, shutil

os.makedirs(EXPORT_LATEST_DIR,  exist_ok=True)
os.makedirs(EXPORT_ARCHIVE_DIR, exist_ok=True)

export_date  = now_local().date()
latest_path  = f"{EXPORT_LATEST_DIR}/{EXPORT_FILE_NAME}"
archive_path = f"{EXPORT_ARCHIVE_DIR}/{EXPORT_FILE_NAME.replace('.csv', '')}_{export_date}.csv"

# utf-8-sig so Excel shows non-English characters correctly.
# An empty result still writes a header-only file, so SharePoint never shows yesterday's data as today's.
pdf.to_csv(latest_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
shutil.copyfile(latest_path, archive_path)
print(f"Written: {latest_path}")
print(f"Archive: {archive_path}")

In [ ]:
# Step 4: delete archive copies older than EXPORT_ARCHIVE_DAYS
from datetime import timedelta

cutoff = export_date - timedelta(days=EXPORT_ARCHIVE_DAYS)
prefix = EXPORT_FILE_NAME.replace(".csv", "") + "_"

for f in os.listdir(EXPORT_ARCHIVE_DIR):
    if f.startswith(prefix) and f.endswith(".csv"):
        try:
            file_date = datetime.strptime(f[len(prefix):-4], "%Y-%m-%d").date()
            if file_date < cutoff:
                os.remove(f"{EXPORT_ARCHIVE_DIR}/{f}")
                print(f"Deleted old archive: {f}")
        except ValueError:
            pass   # not one of our dated files

In [ ]:
# Step 5: return the row count to the pipeline
notebookutils.notebook.exit(str(len(pdf)))